# CUDA Batch Image Processing Pipeline
**GPU-accelerated grayscale → Gaussian blur → Sobel edge detection using CUDA NPP**

Make sure: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi

In [ ]:
# Cell 2: Clone your repo (replace with your actual GitHub URL)
REPO_URL = 'https://github.com/YOUR_USERNAME/cuda-batch-imgproc'  # <-- update this

!git clone {REPO_URL} /content/cuda_batch_imgproc
%cd /content/cuda_batch_imgproc

In [ ]:
# Cell 3: Install dependencies and fetch stb headers
!apt-get install -q -y imagemagick python3-pil
!pip install -q Pillow numpy
!chmod +x scripts/*.sh
!bash scripts/download_stb.sh

In [ ]:
# Cell 4: Download 100 sample/synthetic images
!bash scripts/download_images.sh

In [ ]:
# Cell 5: Build the project
# T4 is sm_75; Colab A100 is sm_80 — Makefile covers both
!make all 2>&1

In [ ]:
# Cell 6: Run the pipeline on all images
!./bin/cuda_imgproc \
    --input data/input \
    --output data/output \
    --log logs/run.log \
    --blur-size 5

In [ ]:
# Cell 7: Show the log (this is your proof of execution)
with open('logs/run.log') as f:
    print(f.read())

In [ ]:
# Cell 8: Visualize a few input/output pairs
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

in_dir  = 'data/input'
out_dir = 'data/output'

in_imgs  = sorted([f for f in os.listdir(in_dir)  if f.endswith('.png')])[:4]
out_imgs = sorted([f for f in os.listdir(out_dir) if f.endswith('.png')])[:4]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('CUDA NPP Pipeline: Input (top) vs Edge-Detected Output (bottom)', fontsize=14)

for i, (inf, outf) in enumerate(zip(in_imgs, out_imgs)):
    axes[0][i].imshow(Image.open(os.path.join(in_dir, inf)))
    axes[0][i].set_title(inf[:20], fontsize=8)
    axes[0][i].axis('off')
    axes[1][i].imshow(Image.open(os.path.join(out_dir, outf)), cmap='gray')
    axes[1][i].set_title(outf[:20], fontsize=8)
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig('logs/comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: logs/comparison.png')

In [ ]:
# Cell 9: Commit logs + outputs back to repo (proof of execution)
!git config user.email 'you@example.com'
!git config user.name 'Your Name'
!git add logs/ data/output/
!git commit -m 'Add execution logs and processed outputs'
# !git push  # uncomment after adding your GitHub token / SSH key